In [19]:
import sys
import os

import pandas as pd
import scanpy as sc
import numpy as np
import warnings

import torch
import torch.nn.functional as F
import torch.nn as nn

from torch.utils.data import DataLoader, TensorDataset
from torch.nn import DataParallel
import matplotlib.pyplot as plt
from matplotlib.pyplot import rc_context
import anndata
import seaborn as sns
import matplotlib.font_manager
from matplotlib import rcParams

font_list = []
fpaths = matplotlib.font_manager.findSystemFonts()
for i in fpaths:
    try:
        f = matplotlib.font_manager.get_font(i)
        font_list.append(f.family_name)
    except RuntimeError:
        pass

font_list = set(font_list)
plot_font = 'Helvetica' if 'Helvetica' in font_list else 'FreeSans'
rcParams['font.family'] = plot_font
rcParams.update({'font.size': 10})
rcParams.update({'figure.dpi': 300})
rcParams.update({'figure.figsize': (3,3)})
rcParams.update({'savefig.dpi': 500})
warnings.filterwarnings('ignore')

%load_ext autoreload
%autoreload 2
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [20]:
# import scanpy as sc
# import pandas as pd

# mtx = "/GSE133344_filtered_matrix.mtx.gz"
# genes = "/GSE133344_filtered_genes.tsv.gz"
# barcodes = "/GSE133344_filtered_barcodes.tsv.gz"
# meta = "/GSE133344_filtered_cell_identities.csv.gz"

# # 1) read matrix (genes x cells)
# adata = sc.read_mtx(mtx).T   # transpose -> cells x genes

# # 2) add barcodes as obs_names
# bc = pd.read_csv(barcodes, header=None, sep="\t")[0].astype(str).values
# adata.obs_names = bc

# # 3) add genes as var_names
# g = pd.read_csv(genes, header=None, sep="\t")
# # common layouts: [gene_id, gene_name] or just one column
# if g.shape[1] >= 2:
#     adata.var["gene_ids"] = g.iloc[:, 0].astype(str).values
#     adata.var_names = g.iloc[:, 1].astype(str).values
# else:
#     adata.var_names = g.iloc[:, 0].astype(str).values
# adata.var_names_make_unique()

# # 4) merge cell identities into obs
# meta_df = pd.read_csv(meta)
# # guess the barcode column name
# barcode_col = None
# for c in ["barcode", "barcodes", "cell", "cell_id", "cellid", "Cell", "CellID"]:
#     if c in meta_df.columns:
#         barcode_col = c
#         break
# if barcode_col is None:
#     # fallback: assume first column is barcode
#     barcode_col = meta_df.columns[0]

# meta_df[barcode_col] = meta_df[barcode_col].astype(str)
# meta_df = meta_df.set_index(barcode_col)

# adata.obs = adata.obs.join(meta_df, how="left")

# # 5) save
# adata.write_h5ad("/GSE133344_filtered.h5ad")
# print(adata)


In [21]:
import scanpy as sc

adata_raw = sc.read_h5ad('GSE133344_filtered.h5ad')

# Function to assign Group based on guide content (case-insensitive)
def assign_group(guide_id):
    guide_lower = guide_id.lower()
    
    has_ptpn12 = 'ptpn12' in guide_lower
    has_zbtb25 = 'zbtb25' in guide_lower
    has_ctrl = 'negctrl' in guide_lower or 'ctrl' in guide_lower
    
    # Assign to specific groups
    if has_ptpn12 and has_zbtb25:
        return 'PTPN12+ZBTB25'
    elif has_ptpn12 and has_ctrl:
        return 'PTPN12+ctrl'
    elif has_zbtb25 and has_ctrl:
        return 'ZBTB25+ctrl'
    elif has_ctrl and not has_ptpn12 and not has_zbtb25:
        return 'ctrl'
    else:
        return 'other'

# Create Group column
adata_raw.obs['Group'] = adata_raw.obs['guide_identity'].apply(assign_group)

# Filter to keep only desired groups
adata_raw = adata_raw[adata_raw.obs['Group'].isin([
    'ctrl',
    'PTPN12+ZBTB25',
    'ZBTB25+ctrl',
    'PTPN12+ctrl'
])].copy()

print(f"Filtered to {adata_raw.n_obs} cells")
print(adata_raw.obs['Group'].value_counts())

Filtered to 69989 cells
Group
ctrl             68501
ZBTB25+ctrl        740
PTPN12+ctrl        445
PTPN12+ZBTB25      303
Name: count, dtype: int64


In [22]:
# Separate ctrl and non-ctrl
adata_ctrl = adata_raw[adata_raw.obs['Group'] == 'ctrl']
adata_other = adata_raw[adata_raw.obs['Group'] != 'ctrl']

print("Original ctrl cells:", adata_ctrl.n_obs)

# Randomly select 7000 ctrl cells (reproducible)
np.random.seed(42)
selected_idx = np.random.choice(adata_ctrl.n_obs, size=6705, replace=False)

adata_ctrl_down = adata_ctrl[selected_idx].copy()

# Combine back
adata_balanced = adata_ctrl_down.concatenate(adata_other)

print("After downsampling:")
print(adata_balanced.obs['Group'].value_counts())

Original ctrl cells: 68501
After downsampling:
Group
ctrl             6705
ZBTB25+ctrl       740
PTPN12+ctrl       445
PTPN12+ZBTB25     303
Name: count, dtype: int64


In [23]:
# adata_raw = sc.read_h5ad('GSE133344_filtered.h5ad')
# #adata = adata[adata.obs['condition']=='ctrl'].copy()#anndata.AnnData(count)
# print(adata_raw.obs)
# adata_raw.obs['Group']=adata_raw.obs['guide_identity']
# print(adata_raw.obs['Group'].unique())
# adata_raw = adata_raw[adata_raw.obs['Group'].isin(['ctrl',
#                                                        'PTPN12+ZBTB25',
#                                                        'ZBTB25+ctrl',
#                                                        'PTPN12+ctrl'])].copy()

In [24]:
adata = adata_balanced.copy()
print(adata)
condi_df = adata.obs['Group'].map({'ctrl':0,
                                     'PTPN12+ctrl':1,
                                     'ZBTB25+ctrl':2,
                                     'PTPN12+ZBTB25':3
                             })
del adata.obs
adata.obs['Group'] = condi_df

AnnData object with n_obs × n_vars = 8193 × 33694
    obs: 'guide_identity', 'read_count', 'UMI_count', 'coverage', 'gemgroup', 'good_coverage', 'number_of_cells', 'Group', 'batch'
    var: 'gene_ids'


In [25]:
adata_ctrl = adata[adata.obs['Group'].isin([0])].copy()

In [26]:
print(adata_ctrl)

AnnData object with n_obs × n_vars = 6705 × 33694
    obs: 'Group'
    var: 'gene_ids'


In [27]:
import numpy as np
import scanpy as sc
import scipy.sparse as sp

# 0) Quick check: where are the bad values?
X = adata.X
if sp.issparse(X):
    n_bad = int((~np.isfinite(X.data)).sum())
    print("X is sparse. non-finite in X.data =", n_bad)
    if n_bad > 0:
        # replace inf/nan in sparse data
        X.data[~np.isfinite(X.data)] = 0.0
        X.eliminate_zeros()
        adata.X = X
else:
    n_bad = int((~np.isfinite(X)).sum())
    print("X is dense. non-finite in X =", n_bad)
    if n_bad > 0:
        adata.X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

# 1) OPTIONAL but recommended:
# HVG should be done on normalized + log1p data (not raw huge values, not scaled z-scores)
# If you already did normalize_total+log1p, you can skip these two lines.
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# 2) Safety check again after log1p
X = adata.X
if sp.issparse(X):
    n_bad = int((~np.isfinite(X.data)).sum())
    print("After log1p: non-finite in X.data =", n_bad)
    if n_bad > 0:
        X.data[~np.isfinite(X.data)] = 0.0
        X.eliminate_zeros()
        adata.X = X
else:
    n_bad = int((~np.isfinite(X)).sum())
    print("After log1p: non-finite in X =", n_bad)
    if n_bad > 0:
        adata.X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

# # 3) Now HVG will run
# sc.pp.highly_variable_genes(adata, n_top_genes=1000, flavor="seurat")
# print("HVG computed. #HVG =", int(adata.var["highly_variable"].sum()))


X is sparse. non-finite in X.data = 0
After log1p: non-finite in X.data = 0


In [28]:
# # make sure highly_variable exists
# sc.pp.highly_variable_genes(adata, n_top_genes=201)

# genes_to_force = [
#     'ALAS2','HBA1','HBA2','HIST1H1C','GYPB','SLC25A37','LGALS1'
# ]

# # force these genes (if present) to be HVG
# present = adata.var_names.isin(genes_to_force)
# print("Forced present genes:", adata.var_names[present].tolist())

# adata.var.loc[present, "highly_variable"] = True

# # subset
# adata = adata.copy()


In [29]:
adata

AnnData object with n_obs × n_vars = 8193 × 33694
    obs: 'Group'
    var: 'gene_ids'
    uns: 'log1p'

In [30]:
# adata.X = adata.X.toarray()

In [31]:
adata_ctl = adata[adata.obs['Group'].isin([0,1,2])].copy()
adata_val = adata[adata.obs['Group'].isin([3])].copy()

In [32]:
adata_val

AnnData object with n_obs × n_vars = 303 × 33694
    obs: 'Group'
    var: 'gene_ids'
    uns: 'log1p'

In [33]:
adata_ctl

AnnData object with n_obs × n_vars = 7890 × 33694
    obs: 'Group'
    var: 'gene_ids'
    uns: 'log1p'

In [34]:
adata_ctl.write('gears_train_data.h5ad')
adata_val.write('gears_test_data.h5ad')

In [35]:
sc.read_h5ad('gears_train_data.h5ad')

AnnData object with n_obs × n_vars = 7890 × 33694
    obs: 'Group'
    var: 'gene_ids'
    uns: 'log1p'

In [36]:
adata_ctl.to_df()

,RP11-34P13.3,FAM138A,OR4F5,RP11-34P13.7,RP11-34P13.8,RP11-34P13.14,RP11-34P13.9,FO538757.3,FO538757.2,AP006222.2,...,AC007325.2,BX072566.1,AL354822.1,AC023491.2,AC004556.1,AC233755.2,AC233755.1,AC240274.1,AC213203.1,FAM231B
ACCGTAATCTGCGGCA-6-0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.475115,0.000000,...,0.0,0.0,0.0,0.0,0.475115,0.0,0.0,0.0,0.0,0.0
GCATGCGGTCGACTAT-7-0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
CTCTACGTCCTCTAGC-5-0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.432551,0.0,0.0,0.0,0.0,0.0
ACAGCCGAGTATTGGA-4-0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.681070,0.397430,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
GTCGTAACACCTCGTT-5-0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.522310,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTGCCGTCACGGTTTA-8-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.566527,0.0,0.0,0.0,0.0,0.0
TTGCGTCAGCCGCCTA-8-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.567220,0.000000,...,0.0,0.0,0.0,0.0,1.190911,0.0,0.0,0.0,0.0,0.0
TTGGAACAGCTAACAA-8-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.314282,0.553053,...,0.0,0.0,0.0,0.0,0.553053,0.0,0.0,0.0,0.0,0.0
TTGGAACAGTTAGCGG-8-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.756936,0.0,0.0,0.0,0.0,0.0
